# Pretrained: zero-shot validation

Подключите исходный **full_psg_v1** и выберите Accelerator **GPU T4 ×2**, Internet On. Используется существующий Secret `GITHUB_TOKEN`. Табличные кэши не нужны. Сначала Run All с `SMOKE_ONLY=True`; после проверки smoke — `False`. Ни обучения, ни подбора параметров нет.

016 — AnySleep run1, 017 — U-Sleep run1 с голосованием двух пар, 018 — SleepFMStager с BAS-only. Необязательный 019 — среднее вероятностей AnySleep run1/2/3. Новые ячейки выводят один обновляемый индикатор на модель.

AnySleep/U-Sleep обучались в том числе на Sleep-EDF: это проверка переноса готового checkpoint на поставку курса, а не доказательство качества на неизвестных checkpoint испытуемых. Перекрывающиеся блоки ограничивают глобальный контекст; smoke сохраняет диагностику чувствительности к границам. Метрики smoke не входят в experiments.csv.

Smoke проверяет одну запись, включая рабочий размер блока и peak VRAM. Короткая диагностика сравнивает цельный и блочный проход; её расхождения сохраняются в boundary_check.json. После успешного GPU smoke переключите SMOKE_ONLY=False для всей validation.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = "feat/pretrained"  # Для полного прогона укажите опубликованный commit SHA.
PROJECT_ROOT = Path("/tmp/sleep-edf-pretrained-code")
OUTPUT_ROOT = Path("/kaggle/working/artefacts")
CHECKPOINT_CACHE = Path("/kaggle/working/pretrained-checkpoints")
CHECKPOINT_ROOT = None  # Необязательно: Input с плоскими файлами весов/исходников.
RESUME_ROOT = None  # Необязательно: распакованный artefacts из прошлого запуска.
CONFIG_IDS = ["016_pretrained", "017_pretrained", "018_pretrained"]
# Добавьте "019_pretrained" для равного ансамбля трёх AnySleep checkpoints (~3× инференс).
SMOKE_ONLY = True  # False: сначала smoke каждой модели, затем её полный val.
DEVICE = "cuda:0"  # На T4×2 можно независимо запустить другую модель на cuda:1.
THREADS = max(1, (os.cpu_count() or 2) - 1)


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import run_command  # noqa: E402

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-pretrained-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.pretrained",
]
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware('"
        + DEVICE
        + "'),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)

In [ ]:
roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("labels.csv")
    if (p.parent / "epochs/val").is_dir() and (p.parent / "MANIFEST.json").is_file()
]
if len(roots) != 1:
    raise RuntimeError(
        f"Нужен один Input full_psg_v1 с epochs/val, labels.csv и MANIFEST.json; найдено: {roots}"
    )
DATA_ROOT = roots[0]
if RESUME_ROOT is not None and not OUTPUT_ROOT.exists():
    shutil.copytree(Path(RESUME_ROOT), OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Данные:", DATA_ROOT, "Результаты:", OUTPUT_ROOT, "SMOKE_ONLY:", SMOKE_ONLY)

In [ ]:
failures = []
for experiment_id in CONFIG_IDS:
    command = [
        *BASE,
        str(PROJECT_ROOT / "configs" / f"{experiment_id}.yaml"),
        "--data-root",
        str(DATA_ROOT),
        "--output-root",
        str(OUTPUT_ROOT),
        "--checkpoint-cache",
        str(CHECKPOINT_CACHE),
        "--device",
        DEVICE,
        "--threads",
        str(THREADS),
    ]
    if CHECKPOINT_ROOT is not None:
        command += ["--checkpoint-root", str(CHECKPOINT_ROOT)]
    try:
        run_command([*command, "--smoke"], cwd=PROJECT_ROOT, env=ENV)
        diagnostic = OUTPUT_ROOT / "smoke" / experiment_id / "boundary_check.json"
        if diagnostic.exists():
            print(
                experiment_id,
                "изменение argmax в проверке границ:",
                json.loads(diagnostic.read_text())["argmax_disagreement"],
            )
        if not SMOKE_ONLY:
            run_command(command, cwd=PROJECT_ROOT, env=ENV)
    except Exception as error:
        failures.append({"experiment": experiment_id, "error": str(error)})
        print(experiment_id, "ошибка; следующий эксперимент продолжается.")
(OUTPUT_ROOT / "notebook_failures.json").write_text(
    json.dumps(failures, indent=2), encoding="utf-8"
)
for experiment_id in CONFIG_IDS:
    folder = (
        OUTPUT_ROOT / "smoke" / experiment_id
        if SMOKE_ONLY
        else OUTPUT_ROOT / experiment_id
    )
    path = folder / "metrics.json"
    if path.exists():
        m = json.loads(path.read_text())
        print(
            experiment_id,
            "SMOKE" if m["smoke"] else "VAL",
            "macro-F1:",
            round(m["validation"]["macro_f1"], 5),
            "сек:",
            round(m["inference_seconds"], 1),
        )
print("Ошибок:", len(failures))

Завершённые записи восстанавливаются автоматически при повторе с тем же кодом/конфигом/данными. Для новой сессии можно подключить прошлый OUTPUT_ROOT как Input и задать RESUME_ROOT. ZIP хранит итоговые вероятности завершённых моделей и промежуточные записи незавершённых; веса и окружение остаются за его пределами. Ошибка одной модели не удаляет результаты других.


In [ ]:
from datetime import UTC, datetime

from IPython.display import FileLink, display

archive = Path("/kaggle/working") / (
    "sleep-edf-pretrained-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S") + ".zip"
)
run_command(
    [*BASE, "--output-root", str(OUTPUT_ROOT), "--export", str(archive)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
display(
    FileLink(
        str(archive.relative_to(Path.cwd())),
        result_html_prefix="Все результаты одним ZIP: ",
    )
)